<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
<center > المؤلف: دينيس ميرونوف (@dmironov)



# <center>توقع عوائد الأسهم مع ARIMA، والأسعار مع Ridge وLasso </center>



## 1. شرح الميزات والبيانات



### 1.1 وصف المهمة



نقوم في المشروع بتحليل خمسة أصول: 
                                                              
    - شركة باريك جولد (ABX) للصناعات الأساسية
    - شركة وول مارت (WMT) لخدمات المستهلك
    - شركة كاتربيلر (CAT) السلع الرأسمالية
    - بي بي بي.إل.سي. (بي بي) الطاقة
    - شركة فورد للسيارات (F) السلع الرأسمالية 
    - شركة جنرال إلكتريك (GE) للطاقة 
    
استنادًا إلى بيانات Yahoo Finance (https://finance.yahoo.com). من أجل إعادة إنتاج نتائج المشروع، يجب إما جمع بيانات الأصول يدويًا من Yahoo Finance للفترة الزمنية اليومية من 10 ديسمبر 2013 إلى 7 ديسمبر 2018 أو تنزيلها هنا https://github.com/dmironov1993/Dataفي هذا المشروع، هدفنا هو التنبؤ بعائدات الأسهم باستخدام نموذج الانحدار الذاتي المتحرك المتكامل (ARIMA) وأسعار الأسهم باستخدام Ridge وLasso. يتم تعريف نموذج ARIMA بواسطة ثلاث معلمات $(p,d,q)$، حيث $p$ - ترتيب نموذج الانحدار الذاتي، $d$ - درجة الاختلاف و$q$ - ترتيب نموذج المتوسط ​​المتحرك. يتم اختيار هذه المعلمات عن طريق القوة الغاشمة (بحث الشبكة) واختيار النموذج ذو أدنى معيار لمعلومات Akaike. إذا كانت السلاسل الزمنية ثابتة، فسيتبقى لدينا معلمات p وq، بينما $d=0$. عادةً ما يُطلق على هذا النموذج اسم ARMA، وسنستخدم هذا الاختصار فيما يلي. ومع ذلك، نظرًا لأن Python لا تحتوي على العديد من المكتبات المهمة مثل تلك الموجودة في R، على سبيل المثال مكتبات التحليل المتزامن ARMA-GARCH، فسيتم التنبؤ بأسعار الأسهم أيضًا باستخدام خوارزميات التعلم الآلي (ML) مثل Ridge و Lasso. في المرحلة المناسبة من عملنا، سنقوم بإنشاء ميزات إضافية. سيتم ضبط خوارزميات تعلم الآلة باستخدام البحث الشبكي للمعلمات الفائقة. سيتم اختبار أداء خوارزميات تعلم الآلة على عينة صغيرة. 



### 1.2. المكتبات ومجموعة البيانات



نقوم هنا باستيراد المكتبات وتحميل البيانات في نموذج CSV الذي سنقوم بتحليله والعمل معه.


In [ ]:
# Jupiter notebook setup and Importing libraries 
# By default, all figures are shown in 'png'. If the latter is changed to 'svg', higher quality is guaranteed
%config InlineBackend.figure_format = 'png'
import warnings
warnings.simplefilter('ignore')

# Data manipulations
import pandas as pd
import numpy as np

# Visualization
import seaborn as sns
from matplotlib import pyplot as plt
%matplotlib inline
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly
import plotly.graph_objs as go
from plotly import tools
import plotly.plotly as py
init_notebook_mode(connected=True)

# ARIMA (ARMA) modelling
import statsmodels.api as sm
import statsmodels.tsa.api as smt
import statsmodels.tsa.stattools as ts

# Statistics
import scipy.stats as scs
from scipy.stats import skew
from scipy.stats import kurtosis
from statsmodels.tsa.stattools import kpss

# Ljung-box test (to check whether residuals are white noise)
from statsmodels.stats.diagnostic import acorr_ljungbox


# Metrics for ML (ARIMA/ARMA has embedded AIC criterion)
from sklearn.metrics import mean_absolute_error

# Hyperparameter tuning and validation
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit 
from sklearn.preprocessing import StandardScaler

# Machine learning algorithms
#from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.linear_model import Ridge, Lasso

In [ ]:
# Symbols of assets
asset_names = ['ABX','WMT','CAT','BP','F','GE']

In [ ]:
# function for loading csv
def read_csv(symbols):
    
    """ 
        reading csv file
        
        Input: list  
           - symbols of traded stocks 
    
        Output: tuple
           - dataframes of traded stocks
    
    """
    
    ListofAssets_df = []
    for asset in symbols:
        ListofAssets_df.append(pd.read_csv('%s.csv' % asset, sep=',')\
                                  .rename(columns={'Adj Close': '%s_Adj_close' % asset})\
                                  .sort_values(by='Date', ascending=False))
        
    return tuple(ListofAssets_df)

In [ ]:
# number of dataframes within read_csv
print (len(read_csv(asset_names)))

In [ ]:
# number of rows and columns within each df
for df in read_csv(asset_names):
    print (df.shape)

In [ ]:
# view of one of dataframes
read_csv(asset_names)[0].head(2)

In [ ]:
# information about structure of data
read_csv(asset_names)[0].info()


نرى أنه لا توجد قيم مفقودة في مجموعات البيانات التي تهمنا.



في الأسعار التاريخية التي تم الحصول عليها، لدينا المعلومات التالية لكل أصل من الأصول:- **التاريخ**: التاريخ
 - **الفتح**: السعر المفتوح خلال تاريخ محدد
 - **مرتفع**: أعلى سعر خلال تاريخ معين
 - **منخفض**: أدنى سعر خلال تاريخ معين
 - **الإغلاق**: سعر الإغلاق خلال تاريخ محدد
 - **`NAME`_Adj_Close**: سعر الإغلاق المعدل في نهاية التاريخ
 - **الحجم**: حجم التداول
 
بالإضافة إلى هذه المعلومات، نحن مهتمون بالعوائد اليومية للأصول. هنا سيتم حساب الأخير باستخدام
   ### <center> $ r^{(i)}_{t} := ln\left(\frac{P^{(i)}_{t}}{P^{(i)}_{t-1}} \right) = ln\left( P^{(i)}_{t}\right) - ln\left( P^{(i)}_{t-1} \right),$ </center>
حيث $ln$ يشير إلى اللوغاريتم الطبيعي، و$P^{(i)}_{t}$ هو سعر الإغلاق المعدل للأصل ${i}$ في اللحظة الزمنية ${t}$، بينما $P^{(i)}_{t-1}$ هو في اللحظة الزمنية السابقة: ${t-1}$.


In [ ]:
# function for adding log-returns column to dataframes 
def add_log_returns(assets_df, symbols):
    
    """ 
        Calculating returns of the assets
        
        Input:
            - assets_df: is a tuple of dataframes
            - symbols: list with symbols of traded stocks in the same order as those in assets_df
        Output:
            - tuple of dataframes with returns, dataframe's index is Date now, 
              dataframe is also sorted by index in ascending order.
              
    """
    
    ListofAssets_df = []
    num_asset = 0
    
    if len(assets_df) == len(symbols):
        
        for df in assets_df:
            adj_closing_price = df['%s_Adj_close' % symbols[num_asset]]
            log_array = np.log(np.array(adj_closing_price))
            log_return_array = log_array - np.append(log_array[1:], np.nan)
            log_return_df = pd.DataFrame(log_return_array,
                                         columns=['%s_returns' % symbols[num_asset]])
            df = df.reset_index().drop(columns=['index'])
            df = pd.concat((df, log_return_df), axis=1)
            df['Date'] = df['Date'].apply(pd.to_datetime)
            df = df.set_index('Date')
            df = df.sort_index(ascending=True)
            df.dropna(axis=0, inplace=True)
            ListofAssets_df.append(df)
            num_asset += 1
            
        return tuple(ListofAssets_df)
    
    else:
        print ('Number of DataFrames and Number of assets considered should be equal')

In [ ]:
asset_dfs = add_log_returns(read_csv(asset_names), asset_names)

In [ ]:
# we have added returns, year, month and day columns
for df in asset_dfs:
    print (df.shape)

In [ ]:
asset_dfs[0].head(3)


الآن، يحتوي كل إطار من إطارات البيانات، بالإضافة إلى الأعمدة الستة المقدمة بالفعل مثل Open وHigh وLow وClose وNAME_Adj_Close وVolume، على عمود إضافي واحد يتعلق بالعائدات. لاحظ أن التاريخ أصبح فهرسًا الآن وليس عمودًا. تم فرز Dataframes حسب التاريخ بترتيب تصاعدي. هناك حاجة إلى هذا الأخير أيضًا للتحقق من صحة السلاسل الزمنية الصحيحة. تم تقليل عدد الصفوف إلى 1257 (1258 في الأصل) نظرًا لأن بياناتنا لا تسمح بحساب العوائد قبل 11 ديسمبر 2013.



للراحة، نقدم وظيفتين. أحدهما لإنشاء إطار بيانات يتكون من أسعار الإغلاق المعدلة لأصولنا فقط، بينما الآخر هو إنشاء إطار بيانات يتكون من عوائد الأصول فقط.


In [ ]:
def asset_adj_close(list_of_df, symbols):
    """
    
        Input:
            - list_of_df: list of dataframes. Each of the latter should have 
              a column corresponding to adjusted close price
            - symbols: list of asset symbols taken from a stock market
        Output: 
            - pandas dataframe consisting of only adjusted close prices of considered assets
    
    """
    
    adj_close = []
    number = 0
    for asset in asset_names:
        df = list_of_df[number]['%s_Adj_close' % asset]
        adj_close.append(df)
        number += 1
        
    return pd.concat(adj_close, axis=1)

In [ ]:
adj_close = asset_adj_close(asset_dfs, asset_names)
adj_close.head(2)

In [ ]:
def asset_returns(list_df, symbols):
    """
    
        Input: 
            - list_df: list of dataframes each of which contains asset returns column
            - symbols: list of asset symbols taken from a stock market
        Output: 
            - a dataframe consisting of only asset returns

    """
    
    asset_returns = []
    k = 0
    for asset in symbols:
        asset_returns.append(list_df[k]['%s_returns' % asset])
        k += 1

    return pd.concat(asset_returns, axis=1)

In [ ]:
returns = asset_returns(asset_dfs, asset_names)

In [ ]:
returns.head(2)


بعد ذلك، نقوم بإجراء تحليل أولي للبيانات المرئية لأسعار الإغلاق والعوائد المعدلة.



## 2. تحليل البيانات الأولية، وتحليل البيانات المرئية الأولية، والرؤى والتبعيات الموجودة



هيكل هذا القسم على النحو التالي- الارتباط وPairplot
- مخطط كمي عادي ومقارنة تقدير كثافة النواة (KDE) بأقرب توزيع طبيعي حدودي
- التواء، التفرطح، القيمة القصوى، القيمة الدنيا، المتوسط والتباين
- مؤامرة مربعة
- مؤامرة السلاسل الزمنية
- وظيفة الارتباط ووظيفة الارتباط الجزئي



###2.1.  الارتباط وPairplot



نقدم هنا جدول ارتباط ومؤامرة مبعثرة مقابل بعضها البعض لكل من الأسعار والعوائد لشركة Barrick Gold Corporation (ABX)، وWalmart Inc. (WMT)، وCaterpillar Inc (CAT)، وBP p.l.c. (BP)، وشركة فورد للسيارات (F)، وشركة جنرال إلكتريك (GE). تتيح لنا مقارنة ارتباطات بيرسون وسبيرمان معرفة مدى تأثير القيم المتطرفة الكبيرة على الصورة العامة لحركة الأصول.



أولا نقدم وظيفتين مساعدتين


In [ ]:
def corr_plot(df, symbols, days=252, title='Returns'):
    
    """
        Input: 
            - df: a dataframe consisting only of data which correlations will be plotted
            - symbols: a list of the companies stocks names
            - days: represent the number of days to the past, set it to 0 to get consider the whole range
            - title: the plot title in accordance with df
        Output: 
            - illustration of assets correlations
    
    """
    
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20,8))
    sns.heatmap(df[-days:].corr(method='pearson'), annot=True, fmt='.2f', cmap="YlGnBu",
                xticklabels=symbols, yticklabels=symbols, ax=axes[0])
    sns.heatmap(df[-days:].corr(method='spearman'), annot=True, fmt='.2f', cmap="YlGnBu",
                xticklabels=symbols, yticklabels=symbols, ax=axes[1])
    axes[0].set_title('Pearson correlation (%s)' % title)
    axes[1].set_title('Spearman correlation (%s)' % title)
    plt.show()

In [ ]:
def pairplot(df, days=252, width=9, height=9):
    
    """
        Input:
            - df: a dataframe consisting only of data which will be plotted
            - width and height: a corresponding size of the plot
        Output: 
            - illustration of assets pairplot
    
    """
    g = sns.pairplot(df[-days:]);
    g.fig.set_size_inches(width,height)
    plt.show()


---



#### 2.1.1 عوائد الأصول 


In [ ]:
corr_plot(returns, asset_names, title='Returns')


من الشكل الموجود أعلاه يمكن للمرء أن يرى أن الارتباط بين عوائد الأصول لا يتغير كثيرًا بين نوعين من الارتباطات. تميل العوائد إلى الانتقال خلال فترة زمنية تبلغ 252 يومًا.



الآن دعونا نلقي نظرة على الصورة الكاملة للارتباطات الزوجية.


In [ ]:
pairplot(returns, width=9, height=9)


#### 2.1.2 أسعار الأصول 


In [ ]:
corr_plot(adj_close, asset_names, title='Adj close price')


في الشكل أعلاه مباشرة، يظهر ارتباط بيرسون لسعر الإغلاق المعدل للأصول على اليسار، في حين أن ارتباط سبيرمان على اليمين. نرى أن بعض الأسعار تميل إلى التحرك بينما يتحرك البعض الآخر في اتجاهين متعاكسين.


In [ ]:
pairplot(adj_close, width=9, height=9)


ما ورد أعلاه هو المخطط المبعثر لأسعار الإغلاق المعدلة. وهذا يسمح لنا برؤية الاتجاه العام لأزواج الأصول.


وفقا لتحليل الارتباط، فإن الأصول التي تم النظر فيها هنا تميل إلى التحرك. ومع ذلك، ينبغي لنا
تذكر أيضًا ما هو الفاصل الزمني الذي يتم اعتباره. هنا نأخذ في الاعتبار فترة زمنية تبلغ 252 يومًا تقابل سنة تداول واحدة تقريبًا. إذا كانت هناك فترة زمنية أخرى ذات أهمية، فإن معاملات الارتباط لجميع الأصول تتغير بشكل مناسب. ليس من الممكن معرفة مسبقاً ما إذا كان الارتباط بين الأصول سيكون أعلى، أو سيبقى كما هو أو أقل مع تغير الفترة الزمنية. لكن مثل هذا التحليل يوضح تقديرًا لقيم الارتباط المحتملة.



###2.2.  مخطط كمي عادي ومقارنة تقدير كثافة النواة (KDE) بأقرب توزيع طبيعي حدودي



في هذا القسم، نناقش العائدات وتوزيعات أسعار الإغلاق المعدلة لشركة United Continental Holdings Inc. (UAL)، BP p.l.c. (BP)، وشركة American Water Works Company Inc. (AWK)، وشركة Ford Motor Company (F)، وشركة General Electric (GE)، وشركة Walmart Inc. (WMT). 



كما سبق أن قدمنا العديد من الوظائف المساعدة أولاً.


In [ ]:
def norm_function(x, mu, sigma):
    return (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-(x-mu)**2 / (2*sigma**2))

In [ ]:
def qqplot_returns(df, symbols, option='returns', size=(10,5), sharex=False, sharey=False, wspace=0.6, hspace=0.4):
    
    """
    
        Input:
              - df:
              - symbols:
              - option: returns or adj_close
              - size:
              - sharex:
              - sharey:
              - wspace:
              - hspace:
              
        
        Output:
              - illustration of a normal quantile-quantile plot of returns
    
    """
    nrows = len(symbols)
    if nrows != 1:
        fig, axes = plt.subplots(nrows=nrows, ncols=2, sharex=sharex, sharey=sharey, figsize=size)
        plt.subplots_adjust(wspace=wspace, hspace=hspace)
        for i in range(nrows):
            
            if option == 'returns':
                name = '%s' % symbols[i]
                name_return = '%s_returns' % symbols[i]
            elif option == 'adj_close':
                name = '%s' % symbols[i]
                name_return = '%s_Adj_close' % symbols[i]
                
            probplot = sm.ProbPlot(df[name_return], dist='norm')  
            fig = probplot.qqplot(line='q', ax=axes[i,0])
            axes[i,0].set_title('Normal Q-Q Plot (%s)' % name)

            sns.distplot(df[name_return], kde=True, hist=False, ax=axes[i,1], color='black')
            count, mean, std, min_, q1, mean, q3, max_ = df[name_return].describe()
            xx = np.linspace(min_, max_, 1000)
            axes[i,1].plot(xx, norm_function(xx, mean, std), '--', color='red')            
            axes[i,1].set_title(name)
            axes[i,1].set_xlabel('')
        plt.show()
    
    else:        
        if option == 'returns':
            name = '%s' % symbols[1]
            name_return = '%s_returns' % symbols[1]
        elif option == 'adj_close':
            name = '%s' % symbols[1]
            name_return = '%s_Adj_close' % symbols[1]
        
        nrows = 1
        fig, axes = plt.subplots(nrows=nrows, ncols=2, sharex=sharex, sharey=sharey, figsize=size)
        plt.subplots_adjust(wspace=wspace, hspace=hspace)
        name = '%s' % symbols[0]
        name_return = '%s_returns' % symbols[0]

        probplot = sm.ProbPlot(df[name_return], dist='norm')  
        fig = probplot.qqplot(line='q', ax=axes[0])
        axes[0].set_title('Normal Q-Q Plot (%s)' % name)

        sns.distplot(df[name_return], kde=True, hist=False, ax=axes[1], color='black')
        count, mean, std, min_, q1, mean, q3, max_ = df[name_return].describe()
        xx = np.linspace(min_, max_, 1000)
        axes[1].plot(xx, norm_function(xx, mean, std), '--', color='red')            
        axes[1].set_title(name)
        axes[1].set_xlabel('')
        plt.show()


---



#### 2.2.1 أسعار الأصول 


In [ ]:
qqplot_returns(adj_close, asset_names, option='adj_close',
               size=(10,20), sharex=False, sharey=False, wspace=0.6, hspace=0.6)


على الجانب الأيسر تظهر المخططات الكمية الكمية العادية، في حين يتم توضيح تقدير كثافة النواة للعائدات وأقرب توزيع طبيعي لها على الجانب الأيمن. الخطوط الحمراء تتوافق مع التوزيع الطبيعي.



نرى أن بعض أسعار الإغلاق المعدلة لها توزيعات ثنائية الوسائط في حين أن البعض الآخر لديه هيكل أكثر تعقيدًا والذي لا يبدو طبيعيًا حتى بالقرب من ذلك.



#### 2.2.1 عوائد الأصول 


In [ ]:
qqplot_returns(returns, asset_names, option='returns',
               size=(10,20), sharex=False, sharey=False, wspace=0.6, hspace=0.6)

الخطوط الحمراء تتوافق مع التوزيع الطبيعي. لاحظ أن التوزيع ليس طبيعيًا كما يتضح من كلا النوعين من المؤامرات التي توضح ذيولًا أكثر بدانة وتفرطحًا أعلى. ومع ذلك، فإن هيكلها أقرب إلى التوزيع الطبيعي من سعر الإغلاق المعدل.



في هذه المرحلة من العمل، قد نستنتج أن ذيول الدهون ستصبح مشكلة بالنسبة لنمذجة ARIMA أو ARMA الخاصة بنا نظرًا لأنه قد لا نشمل جميع معلومات السلاسل الزمنية بسبب ذلك. دعونا نضع هذه الفكرة في الاعتبار ونمضي قدمًا.



###2.3.  التواء، التفرطح، القيمة القصوى، القيمة الدنيا، المتوسط والتباين



الآن دعونا نجمع المزيد من الإحصائيات حول القيم المستهدفة. بالمعنى الدقيق للكلمة، انحرافهم، التفرطح، ماكس العوائد، ماكس الخسارة، المتوسط ​​​​والتباينات.


In [ ]:
def stats(df, symbols):
    """
    
        Input: 
            - symbols: a list of asset symbols
            
        Output:
            - a dataframe containing information such as Skewness, Kurtosis, Max value,
              Min value, Mean and Variance of the df
        
    """
    
    stat = pd.DataFrame(index=asset_names, 
                                columns=['Skewness','Kurtosis','Max value',
                                         'Min value','Mean','Variance'])
    
    stat['Skewness'] = skew(df, axis=0)
    stat['Kurtosis'] = kurtosis(df, axis=0)
    stat['Max value'] = df.agg('max').values
    stat['Min value'] = df.agg('min').values
    stat['Mean'] = df.agg('mean').values
    stat['Variance'] = df.agg('var').values

    return stat

In [ ]:
stats(adj_close, asset_names)

In [ ]:
stats(returns, asset_names)


###2.4.  مؤامرة مربع من العائدات



سوف تساعدنا مؤامرة الصندوق في الحصول على معلومات دقيقة حول القيم المتطرفة وكيفية ملاءمتها للصورة بأكملها.


In [ ]:
def asset_box_plot(df, symbols, title=None, width=700, height=400, 
                   jitter=0.2, pointpos=-1.5, boxpoints = 'suspectedoutliers'):
    """
    
        Input: 
              - df: a dataframe which columns will be plotted, 
              - symbols: a list of symbols in the order the same as that in dataframe.
              - title:
              - width: 
              - height: 
              - jitter: 
              - pointpos: 
              
        Output: 
              - Box plot illustrated by plotly library
    
    """
    
    data=[]
    for i in range(len(symbols)):
        trace = go.Box(y = df.iloc[:,i],
                       name = symbols[i],
                       jitter=jitter,
                       pointpos=pointpos,
                       boxpoints = boxpoints)

        data.append(trace)
        
        
    layout = go.Layout(title = title,
                       autosize=False,
                       width=width,
                       height=height)

    fig = go.Figure(data=data,layout=layout)

    iplot(fig)


### 2.4.1 سعر الأصول


In [ ]:
asset_box_plot(adj_close, asset_names, title='Adj close price', boxpoints = 'suspectedoutliers')


### 2.4.2 عوائد الأصول


In [ ]:
asset_box_plot(returns, asset_names, title='Returns')


### 2.5.  مؤامرات السلاسل الزمنية


In [ ]:
# A function for plotting stock price history of all assets
def plot_prices(df, symbols, width=500, height=300):
    """
    
        Input: First:  list with symbols of traded stocks
               Second: dataframe containing only adjusted close price columns
    
        Output: asset prices plotting with plotly
    
    """
    traces = []
    for asset in symbols:
        trace = go.Scatter(
                    x=df.index,
                    y=df['%s_Adj_close' % asset],
                    name = '%s price' % asset)
        traces.append(trace)
        
    layout = go.Layout(title='Adj close price history', 
                       autosize=False,
                       width=width,
                       height=height)
#    layout = {'title': 'Stocks Price History'}
    fig = go.Figure(data=traces, layout=layout) 
    
    return iplot(fig, show_link=False)


#### 2.5.1 أسعار الأصول 


In [ ]:
plot_prices(adj_close, asset_names, height=600, width=800)


#### 2.5.2 عوائد الأصول 


In [ ]:
# A function for plotting history of log returns for all assets
def plot_returns(df, symbols, width=500, height=300):
    """
    
        Input: First:  list with symbols of traded stocks
               Second: iterator with dataframes of traded stocks
    
        Output: asset log returns plotting with plotly
    
    """
    traces = []
    for asset in symbols:
        trace = go.Scatter(
                    x=df.index,
                    y=df['%s_returns' % asset],
                    name = '%s returns' % asset,
                    opacity=0.8)
        traces.append(trace)
        
    layout = go.Layout(title='Returns history',
              autosize=False,
              width=width,
              height=height)
    fig = go.Figure(data=traces, layout=layout) 
    
    return iplot(fig, show_link=False)

In [ ]:
plot_returns(returns, asset_names)


قد يكون من المفيد رؤية كافة القيم المرجعة بشكل منفصل.


In [ ]:
# A function for plotting history of log returns for all assets
def plot_returns_indiv(df, symbols, width=800, height=600):
    """
        
        Input: First:  list with symbols of traded stocks
               Second: iterator with dataframes of traded stocks
    
        Output: asset log returns plotting individually with plotly
    
    """
    traces = []
    count = 0
    for asset in symbols:
        trace = go.Scatter(
                    x=df.index,
                    y=df['%s_returns' % asset],
                    name = '%s returns' % asset,
                    opacity=0.8)
        traces.append(trace)
        count += 1
        
    fig = tools.make_subplots(rows=int(len(symbols)/2+0.5), cols=2, shared_yaxes=True)

    i = 0
    while i < count:
        for ncol in [1,2]:
            for nrow in range(1, int(len(symbols)/2+1.5)):
                fig.append_trace(traces[i], nrow, ncol)
                i += 1
        
    fig['layout'].update(width=width, height=height, title='Returns History')
 
    return iplot(fig, show_link=False)

In [ ]:
plot_returns_indiv(returns, asset_names);


من أرقام السلاسل الزمنية للعودة يمكن للمرء أن يرى ذلك 
1. لا تحتوي الأصول على انحرافات طويلة المدى عن المتوسط وتتأرجح بشكل رئيسي فوق بعض القيم الثابتة التي تقارب الصفر. هذه هي خاصية العملية الثابتة.
2. هناك فترات من التقلبات العالية تليها فترات من الهدوء النسبي (تجمع التقلبات).
3. يمكن للمرء أن يلاحظ أيضًا أن تجميع التقلبات لا يشير إلى عدم الثبات ولكن يمكن اعتباره نوعًا من الاعتماد في التباين الشرطي لكل سلسلة.


بعد ذلك، سنحدد ما إذا كانت السلاسل الزمنية للسعر والعودة ثابتة من خلال إيجاد وظيفة الارتباط التلقائي (ACF) ووظيفة الارتباط التلقائي الجزئي (PACF).



###2.6.  وظيفة الارتباط التلقائي (ACF) ووظيفة الارتباط التلقائي الجزئي (PACF)



عند ملاحظة سلسلة زمنية، يكون السؤال الطبيعي هو ما إذا كانت تبدو ثابتة. في هذا القسم، قمنا برسم ACF و PACF من أجل التحقق من وجود الثبات في السلسلة. نقوم أيضًا بإجراء اختبار ديكي-فولر المعزز بالإضافة إلى اختبار كوياتكوفسكي-فيليبس-شميت-شين. يختبر الاختبار الأول فرضية العدم القائلة بأن السلاسل الزمنية ليست ثابتة مقابل فرضية بديلة مفادها أن السلاسل الزمنية ثابتة، في حين تم تصميم الاختبار الأخير لاختبار ثبات السلاسل الزمنية مقابل بديل لعدم الثبات.


In [ ]:
def stationary_analysis(df, symbols, lags, option='price', figsize=(10,20), wspace=0.3, hspace=0.5):
    """
        Plot time series, its Autocorrelation Function (ACF) and Partial Autocorrelation function (PACF)
        
        Input: 
              - df: Dataframe where columns represent either price or return values. 
              - symbols:  Whether stationary analysis is perfored for price or return is 
              - lags: regulated by option parameter. The latter should be either 
              - option: 'price' or 'return'
              - figsize:
              - wspace: 
              - hspace:
              
        Output: 
              - Illustration of Autocorrelation function in the right column, 
                and Partial autocorrelation function in the left
    """
    
    nrows = df.shape[1]
    fig, axes = plt.subplots(nrows=nrows, ncols=2, figsize=figsize)
    plt.subplots_adjust(wspace=0.3, hspace=0.5)
    row = 0
    for asset in symbols:
        if option == 'price':
            smt.graphics.plot_acf(df['%s_Adj_close' % asset], lags=lags, ax=axes[row,0])
            smt.graphics.plot_pacf(df['%s_Adj_close' % asset], lags=lags, ax=axes[row,1])
            axes[row,0].set_title('%s Adj close (ACF)' % asset)
            axes[row,1].set_title('%s Adj close (PACF)' % asset)
            row += 1
        elif option == 'return':
            smt.graphics.plot_acf(df['%s_returns' % asset], lags=lags, ax=axes[row,0])
            smt.graphics.plot_pacf(df['%s_returns' % asset], lags=lags, ax=axes[row,1])
            axes[row,0].set_title('%s returns (ACF)' % asset)
            axes[row,1].set_title('%s returns (PACF)' % asset)
            row += 1     
    
    return plt.show()

In [ ]:
def adf_kpss(df, symbols, option='price'):
    
    """
        Input:
            - df: a dataframe containing only either of prices or returns
            - symbols: list of asset symbols to analyze
            - option: either price or returns in accordance with df
    
        Output:
            - a dataframe constaining Augmented Dickey-Fuller (ADF) and Kwiatkowski–Phillips–Schmidt–Shin (KPSS) 
              test results of asset prices or returns 
    
    """
    adf_kpss = pd.DataFrame(index=asset_names, columns=['ADF','KPSS'])
    
    if option == 'price':
        adf_price = []
        kpss_price = []
        for i in range(len(symbols)):
            adf_price.append(ts.adfuller(adj_close.iloc[:,i])[1])
            kpss_price.append(kpss(adj_close.iloc[:,i])[1])
        adf_kpss['ADF'] = np.array(adf_price)
        adf_kpss['KPSS'] = np.array(kpss_price)
    
    elif option == 'returns':
        adf_returns = []
        kpss_returns = []
        for i in range(len(asset_names)):
            adf_returns.append(ts.adfuller(returns.iloc[:,i])[1])
            kpss_returns.append(kpss(returns.iloc[:,i])[1])
        adf_kpss['ADF'] = np.array(adf_returns)
        adf_kpss['KPSS'] = np.array(kpss_returns)
        
    return adf_kpss


---



#### 2.6.1 أسعار الأصول 


In [ ]:
stationary_analysis(adj_close, asset_names, option='price', lags=30, figsize=(20,20))


وظيفة الارتباط التلقائي (ACF) ووظيفة الارتباط التلقائي الجزئي (PACF) لسلاسل زمنية مكونة من ستة أسعار (Barrick Gold Corporation (ABX)، وWalmart Inc. (WMT)، وCaterpillar Inc (CAT)، وBP p.l.c. (BP)، وشركة Ford Motor (F) وشركة General Electric (GE)). يمكن للمرء أن يرى أن كل ACF يظهر نمط اضمحلال خطي بطيء جدًا (بالكاد ملحوظ)، وهو نموذجي لسلسلة زمنية غير ثابتة. علاوة على ذلك، فإن PACF لديه ارتفاع كبير واحد في التأخر 1 (لا يتم حساب التأخر 0) مما يعني أن (تقريبًا) جميع الارتباطات الذاتية ذات الترتيب الأعلى يتم تفسيرها بشكل فعال من خلال الارتباط التلقائي للتأخر 1. يعتبر كل من ACF وPACF نموذجيين للسلاسل الزمنية غير الثابتة. هناك أيضًا بعض إحصائيات الاختبار التي تدعم أو ترفض افتراضاتنا.
لإثبات تخميننا الأولي حول سلوك السلاسل الزمنية للسعر، قمنا بإجراء عملية ديكي فولر المعززة
اختبار (ADF) واختبار كويتكوفسكي-فيليبس-شميت-شين (KPSS).


In [ ]:
adf_kpss_price = pd.DataFrame(index=asset_names, columns=['ADF','KPSS'])

adf_price = []
kpss_price = []
for i in range(len(asset_names)):
    adf_price.append(ts.adfuller(adj_close.iloc[:,i])[1])
    kpss_price.append(kpss(adj_close.iloc[:,i])[1])

In [ ]:
adf_kpss_price['ADF'] = np.array(adf_price)
adf_kpss_price['KPSS'] = np.array(kpss_price)

In [ ]:
adf_kpss_price

يحتوي هذا الجدول على معلومات القيم الاحتمالية حول ما إذا كان سيتم رفض فرضية العدم أم لا. نظرًا لأن اختبارات ADF تعطي قيمًا p أكبر من 0.05، فليس لدينا معلومات كافية لرفض الفرضية الصفرية التي تنص على أن السلاسل الزمنية ليست ثابتة عند مستوى ثقة 95%. علاوة على ذلك، تظهر قيم KPSS أننا نرفض الفرضية الصفرية القائلة بأن السلاسل الزمنية ثابتة عند مستوى ثقة 95٪، ولكن ABX. من المحتمل أن يكون الأخير انحرافًا بسبب عدم كفاية البيانات. بشكل عام، تثبت نتائج اختبار ADF وKPSS أن السلاسل الزمنية لأسعار الإغلاق المعدلة ليست ثابتة.



#### 2.6.2 عوائد الأصول 


In [ ]:
stationary_analysis(returns, asset_names, option='return', lags=20, figsize=(20,20))


تثبت ACF وPACF أن السلاسل الزمنية للعائدات يجب أن تكون ثابتة. دعونا ندعم ذلك من خلال إجراء اختبارات ADF وKPSS.


In [ ]:
adf_kpss_returns = pd.DataFrame(index=asset_names, columns=['ADF','KPSS'])

adf_returns = []
kpss_returns = []
for i in range(len(asset_names)):
    adf_returns.append(ts.adfuller(returns.iloc[:,i])[1])
    kpss_returns.append(kpss(returns.iloc[:,i])[1])

In [ ]:
adf_kpss_returns['ADF'] = np.array(adf_returns)
adf_kpss_returns['KPSS'] = np.array(kpss_returns)

In [ ]:
adf_kpss_returns


من الجدول نرى أنه بالنسبة لجميع الأصول باستثناء GE، قد يتم رفض فرضية ADF الصفرية التي تنص على أن السلاسل الزمنية ليست ثابتة بمستوى ثقة 95٪، في حين تشير القيم الاحتمالية لاختبار KPSS إلى عدم وجود معلومات كافية لرفض فرضيتها الصفرية بأن السلاسل الزمنية ثابتة عند مستوى ثقة 95٪. ثم قد نستنتج أن السلاسل الزمنية للعائدات ثابتة مما يعني أنه يمكننا استخدام ARMA لأصول ABX وWMT وCAT وBP وF. ومع ذلك، يجب أن نكون حذرين مع شركة جنرال إلكتريك. لكي تكون في الجانب الآمن، قمنا بتضمين المعلمة $d$ أثناء إجراء بحث شبكة ARIMA عن GE.



### 3. المقاييس واختيار النموذج



#### 3.1.  نموذج أريما واختبار Box-Ljung



نحن هنا نبني نموذج ARIMA للسلاسل الزمنية الثابتة لعوائد الأصول باتباع الخطوات التالية:1. حدد قيم p وq لـ ARIMA باستخدام البحث الشبكي المتضمن **مقاييس معيار المعلومات Akaike**. 
2. رسم الأشكال والتحقق من توزيع البقايا (سواء كانت لها خصائص الضوضاء البيضاء أم لا)
3. قم بتطبيق اختبار Box-Ljung على بقايا ARIMA (الفرضية الصفرية هي أن البقايا عبارة عن ضوضاء بيضاء)
4. قم بالتنبؤات باستخدام نماذج ARIMA
نستخدم المواد المذكورة من:
1. https://mlcourse.ai/notebooks/blob/master/jupyter_english/topic09_time_series/topic9_part1_time_series_python.ipynb?flush_cache=true
2. http://www.blackarbs.com/blog/time-series-analysis-in-python-linear-models-to-garch/11/1/2016


In [ ]:
# Defined the function to fit ARIMA(p, d, q) model
# pick best order and final model based on aic

def arima_model(df, symbol):

    """
    
        Input: 
    
        Output:
    
    """
    
    asset = df['%s_returns' % symbol]    
    best_aic = np.inf 
    best_order = None
    best_mdl = None

    pq_rng = range(6) # [0,1,2,3,4 5]  (for GE this was extended up to range(8))
    d_rng = range(1) # [0]  (for GE we used range(2))
    
    for p in pq_rng:
        for d in d_rng:
            for q in pq_rng:
                try:
                    tmp_mdl = smt.ARIMA(asset, order=(p,d,q)).fit(method='mle', trend='nc')
                    tmp_aic = tmp_mdl.aic
                    if tmp_aic < best_aic:
                        best_aic = tmp_aic
                        best_order = (p, d, q)
                        best_mdl = tmp_mdl
                except: continue


#    print(' {}: aic: {:6.5f} | order: {}'.format(symbol, best_aic, best_order))
    return best_aic, best_order, best_mdl

In [ ]:
def tsplot(y, lags=None, figsize=(10, 8), style='bmh', name='asset'):
    if not isinstance(y, pd.Series):
        y = pd.Series(y)
    with plt.style.context(style):    
        fig = plt.figure(figsize=figsize)
        #mpl.rcParams['font.family'] = 'Ubuntu Mono'
        layout = (3, 2)
        ts_ax = plt.subplot2grid(layout, (0, 0), colspan=2)
        acf_ax = plt.subplot2grid(layout, (1, 0))
        pacf_ax = plt.subplot2grid(layout, (1, 1))
        qq_ax = plt.subplot2grid(layout, (2, 0))
        pp_ax = plt.subplot2grid(layout, (2, 1))
        
        y.plot(ax=ts_ax)
        ts_ax.set_title('Time Series Analysis Plots: %s' % name, fontsize=20)
        smt.graphics.plot_acf(y, lags=lags, ax=acf_ax, alpha=0.5)
        smt.graphics.plot_pacf(y, lags=lags, ax=pacf_ax, alpha=0.5)
        sm.qqplot(y, line='s', ax=qq_ax)
        qq_ax.set_title('QQ Plot')        
        scs.probplot(y, sparams=(y.mean(), y.std()), plot=pp_ax)

        plt.tight_layout()


---


In [ ]:
# run ARMA model for the asset returns
# save ARMA output as dictionaries
"""
aic_dict = {}
order_dict = {}
mdl_dict = {}
for symbol in asset_names:
aic_dict[symbol], order_dict[symbol], mdl_dict[symbol] = arima_model(returns, symbol)
"""


لتوفير الوقت، نذكر نتائج البحث الشبكي عن ARMA:
- ABX: أرما (3,2)
- ومت: أرما (0,2)
- القط: أرما (1,0)
- قوة المعركة: أرما (5,4)
- ف: أرما (5,5)
- جنرال إلكتريك: أريما(6,1,7)
يعتمد بحث الشبكة على معيار معلومات Akaike. تم تقييد الشكل من أجل العثور على قيم بخيلة، في حين تم استخدام الأخير لأن عوائد السلاسل الزمنية ثابتة كما ثبت أعلاه.


In [ ]:
order_dict = {}
mdl_dict = {}

In [ ]:
order_dict['ABX'] = (3,0,2)
order_dict['WMT'] = (0, 0, 2)
order_dict['CAT'] = (1, 0, 0)
order_dict['BP'] = (5,0,4)
order_dict['F'] = (5, 0, 5)
order_dict['GE'] = (6, 1, 7)

In [ ]:
mdl_dict['ABX'] = smt.ARIMA(returns.ABX_returns, order=(3,0,2)).fit(method='mle', trend='nc')
mdl_dict['WMT'] = smt.ARIMA(returns.WMT_returns, order=(0,0,2)).fit(method='mle', trend='nc')
mdl_dict['CAT'] = smt.ARIMA(returns.CAT_returns, order=(1,0,0)).fit(method='mle', trend='nc')
mdl_dict['BP'] = smt.ARIMA(returns.BP_returns, order=(5,0,4)).fit(method='mle', trend='nc')
mdl_dict['F'] = smt.ARIMA(returns.F_returns, order=(5,0,5)).fit(method='mle', trend='nc')
mdl_dict['GE'] = smt.ARIMA(returns.GE_returns, order=(6,1,7)).fit(method='mle', trend='nc')

In [ ]:
#for symbol in asset_names:
#    print (mdl_dict[symbol].summary())
#    tsplot(mdl_dict[symbol].resid, lags=30)

In [ ]:
print (mdl_dict['ABX'].summary())
tsplot(mdl_dict['ABX'].resid, lags=30, name='ABX')

In [ ]:
print (mdl_dict['WMT'].summary())
tsplot(mdl_dict['WMT'].resid, lags=30, name='WMT')

In [ ]:
print (mdl_dict['CAT'].summary())
tsplot(mdl_dict['CAT'].resid, lags=30, name='CAT')

In [ ]:
print (mdl_dict['BP'].summary())
tsplot(mdl_dict['BP'].resid, lags=30, name='BP')

In [ ]:
print (mdl_dict['F'].summary())
tsplot(mdl_dict['F'].resid, lags=30, name='F')

In [ ]:
print (mdl_dict['GE'].summary())
tsplot(mdl_dict['GE'].resid, lags=30, name='GE')


من هذه الأرقام يمكن للمرء أن يرى أنه لا يوجد أي ارتباط ذاتي تقريبًا في البقايا. يجب التحقق من ذلك عن طريق اختبار Ljung_box



** اختبار جونغ بوكس **


In [ ]:
ljung_box = {}
for symbol in asset_names:
    ljung_box[symbol] = acorr_ljungbox(mdl_dict['GE'].resid, lags=15)[1]

In [ ]:
ljung_box


لا يمكننا رفض الفرضية الصفرية القائلة بأن البقايا عبارة عن ضوضاء بيضاء عند مستوى ثقة 95٪.



** توقعات ARMA **


In [ ]:
# Plot 21 day forecast
n_steps = 21
holdout_signal = {}
for symbol in asset_names:
    plt.style.use('bmh')
    fig = plt.figure(figsize=(20,10))
    ax = plt.gca()

    ts = returns['%s_returns' % symbol].iloc[-255:].copy()
    ts.plot(ax=ax, label='%s Returns' % symbol)
    pred = mdl_dict[symbol].predict()[-255:]
    pred.plot(ax=ax, style='r-', label='In-sample prediction')
    
    holdout_signal[symbol] = np.sign(pred.values)

    styles = ['b-', '0.2', '0.75', '0.2', '0.75']
    f, err95, ci95 = mdl_dict[symbol].forecast(steps=n_steps) # 95% CI
    _, err99, ci99 = mdl_dict[symbol].forecast(steps=n_steps, alpha=0.01) # 99% CI
    idx = pd.date_range(returns.index[-1], periods=n_steps, freq='D')
    fc_95 = pd.DataFrame(np.column_stack([f, ci95]), index=idx, columns=['forecast', 'lower_ci_95', 'upper_ci_95'])
    fc_99 = pd.DataFrame(np.column_stack([ci99]), index=idx, columns=['lower_ci_99', 'upper_ci_99'])
    fc_all = fc_95.combine_first(fc_99)
    fc_all.plot(ax=ax, style=styles)
    plt.fill_between(fc_all.index, fc_all.lower_ci_95, fc_all.upper_ci_95, color='gray', alpha=0.7)
    plt.fill_between(fc_all.index, fc_all.lower_ci_99, fc_all.upper_ci_99, color='gray', alpha=0.2)
    plt.title('{} Day {} Return Forecast\nARIMA{}'.format(n_steps, symbol, order_dict[symbol]))
    plt.legend(loc='best', fontsize=10)


نحن بحاجة إلى جمع المزيد من البيانات لشركة GE من أجل تصميم عوائدها باستخدام ARMA بدلاً من ARIMA.


أثناء التنبؤ بعوائد الأصول، يمكن للمرء أيضًا ملاحظة أن هناك مجموعات من التقلبات مما يعني أننا بحاجة إلى استخدام نموذج الانحدار الذاتي المشروط المتغاير (GARCH). في كثير من الأحيان، يتم تطبيق نموذج GARCH على مخلفات ARIMA لمعالجة هذه المشكلة، في حين أن هذا النهج ليس متسقًا ذاتيًا. من أجل جعل التنبؤات متسقة ذاتيا، يحتاج المرء إلى نمذجة سلسلة العودة باستخدام نموذج ARMA-GARCH. ومع ذلك، لا توجد مثل هذه المكتبة في بايثون. من أجل الحصول على نتيجة أفضل، نحتاج إلى إنشاء نموذج ARMA-GARCH بأنفسنا. ومع ذلك، فهي ليست مهمة تافهة وتتجاوز نطاق هذا العمل. وهذه إحدى الحالات المحتملة لمزيد من تحسين الحل.



لن نقوم بتطوير نهج ARMA-GARCH هنا، ولكن يمكننا التنبؤ بأسعار الأصول باستخدام تقنية التعلم الآلي. سيتم استخدام خوارزميات Ridge و Lasso لهذا الغرض.



### 4. المعالجة المسبقة للبيانات، والتحقق المتبادل، وتعديل المعلمات الفائقة للنموذج، وإنشاء ميزات جديدة



** هناك أيضًا مقاييس واختيار النموذج **



#### 4.1 تعديل توقعات سعر الإغلاق باستخدام Ridge وLasso 



قبل إجراء أي تنبؤات، دعونا أولاً نقدم العديد من الوظائف التي ستكون مفيدة في تصميمنا بالإضافة إلى معلمات الشبكة لـ Ridge وLasso.
يتم أخذ العديد من الوظائف والأساليب من:
https://mlcourse.ai/notebooks/blob/master/jupyter_english/topic09_time_series/topic9_part1_time_series_python.ipynb?flush_cache=true


In [ ]:
# function to split the dataset into train and test
def timeseries_train_test_split(X, y, test_size):
    """
        Perform train-test split with respect to time series structure
    """
    
    # get the index after which test set starts
    test_index = int(len(X)*(1-test_size))
    
    X_train = X.iloc[:test_index]
    y_train = y.iloc[:test_index]
    X_test = X.iloc[test_index:]
    y_test = y.iloc[test_index:]
    
    return X_train, X_test, y_train, y_test

In [ ]:
# function to prepare data.
# original function is presented in mlcourse.ai in lesson 9 part 1 (time-series)
# here its slighly modified version is used
def prepareData(series, lag_start=1, lag_end=20, test_size=0.2):
    
    """
        series: pd.DataFrame
            - dataframe with timeseries

        lag_start: int
            - initial step back in time to slice target variable 
              example - lag_start = 1 means that the model 
                      will see yesterday's values to predict today

        lag_end: int
            - final step back in time to slice target variable
              example - lag_end = 4 means that the model 
                      will see up to 4 days back in time to predict today

        test_size: float
            - size of the test dataset after train/test split as percentage of dataset
        
    """
    
    # copy of the initial dataset
    data = pd.DataFrame(series.copy())
    data.columns = ["y"]
    
    # lags of series
    for i in range(lag_start, lag_end):
        data["lag_{}".format(i)] = data.y.shift(i)
    
    # datetime features
    data['year'] = data.index.year
    data['month'] = data.index.month
    data['day'] = data.index.day
    data['weekday'] = data.index.weekday
    data['season'] = data['month'].apply(lambda x: '1' if x in [12,1,2] else\
                                         ('2' if x in [3,4,5] else ('3' if x in [6,7,8] else '4')))
    
    time_feat = ['year','month','day','weekday','season']
    time_feat_df = pd.DataFrame(data[time_feat], index=data.index)
    time_feat_df[time_feat] = time_feat_df[time_feat].astype('str')
    time_feat_df = pd.get_dummies(time_feat_df)
    data = pd.concat((data, time_feat_df), axis=1)
    data.drop(columns=time_feat, inplace=True)
    
    y = data.dropna().y
    X = data.dropna().drop(['y'], axis=1)
    X_train, X_test, y_train, y_test = timeseries_train_test_split(X, y, test_size=test_size)

    return X_train, X_test, y_train, y_test

In [ ]:
# perform scaling of features
def feature_scaling(series, lag_start=1, lag_end=20, test_size=0.2):
    
    X_train, X_test, y_train, y_test = prepareData(series, lag_start=lag_start, \
                                                   lag_end=lag_end, test_size=test_size)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    return X_train_scaled, X_test_scaled, y_train, y_test

In [ ]:
time_split = TimeSeriesSplit(n_splits=3)

In [ ]:
def grid_search(estimator, X, y, grid_param, scoring='neg_mean_absolute_error', idd=False, cv=time_split,
                figsize=(10,5)):
    
    gsearch = GridSearchCV(estimator = estimator, 
                           param_grid = grid_param, 
                           scoring=scoring,
                           iid=idd, 
                           cv=cv,
                           n_jobs=-1,
                           verbose=True)
    gsearch.fit(X, y)        
    return gsearch

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred): 
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

In [ ]:
def plotResults(model, X_test, y_test, figsize=(15, 7), asset_name = ''):
    
    
    prediction = model.predict(X_test)
    plt.figure(figsize=figsize)
    plt.plot(prediction, "g", label="%s prediction" % asset_name, linewidth=2.0)
    plt.plot(y_test.values, label="actual", linewidth=2.0)
    
    
    error = mean_absolute_percentage_error(y_test, prediction)
    plt.title("Mean absolute percentage error {0:.2f}%".format(error))
    plt.legend(loc="best")
    plt.show()


** المعلمات الفائقة للبحث في الشبكة **


In [ ]:
param_ridge = {'alpha': [25,10,4,2,1.0,0.8,0.5,0.3,0.2,0.1,0.05,0.02,0.01]}

In [ ]:
param_lasso = {'alpha': [25,10,4,2,1.0,0.8,0.5,0.3,0.2,0.1,0.05,0.02,0.01]}


دعونا نتحقق مرة أخرى من أن جميع البيانات مرتبة تصاعديًا


In [ ]:
adj_close.head(2)


---



### 5. التنبؤات للعينات الرافضة



#### 5.1 توقعات ABX


In [ ]:
#X_ABX_train = prepareData(adj_close.ABX_Adj_close)
X_ABX_train_scaled, X_ABX_test_scaled, y_ABX_train, y_ABX_test = feature_scaling(adj_close.ABX_Adj_close)

In [ ]:
[(el[0].shape, el[1].shape) for el in time_split.split(X_ABX_train_scaled)]


#### 5.1.1 أبكس ريدج


In [ ]:
ridge_ABX = grid_search(estimator = Ridge(),
                      X=X_ABX_train_scaled, 
                      y=y_ABX_train,
                      grid_param=param_ridge,
                      cv=time_split)

In [ ]:
plotResults(ridge_ABX, X_ABX_test_scaled, y_ABX_test, figsize=(15, 7), asset_name='ABX')


#### 5.1.2 ايه بي اكس لاسو


In [ ]:
lasso_ABX = grid_search(estimator = Lasso(),
                      X=X_ABX_train_scaled, 
                      y=y_ABX_train,
                      grid_param=param_lasso,
                      cv=time_split)

In [ ]:
plotResults(lasso_ABX, X_ABX_test_scaled, y_ABX_test, figsize=(15, 7), asset_name='ABX')


---



#### 5.2 تنبؤات WMT


In [ ]:
X_WMT_train = prepareData(adj_close.WMT_Adj_close)
X_WMT_train_scaled, X_WMT_test_scaled, y_WMT_train, y_WMT_test = feature_scaling(adj_close.WMT_Adj_close)

In [ ]:
[(el[0].shape, el[1].shape) for el in time_split.split(X_WMT_train_scaled)]

#### 5.2.1 دبليو إم تي ريدج


In [ ]:
ridge_WMT = grid_search(estimator = Ridge(),
                      X=X_WMT_train_scaled, 
                      y=y_WMT_train,
                      grid_param=param_ridge, 
                      cv=time_split)

In [ ]:
plotResults(ridge_WMT, X_WMT_test_scaled, y_WMT_test, figsize=(15, 7), asset_name='WMT')


#### 5.2.2 WMT لاسو


In [ ]:
lasso_WMT = grid_search(estimator = Lasso(),
                      X=X_WMT_train_scaled, 
                      y=y_WMT_train,
                      grid_param=param_lasso, 
                      cv=time_split)

In [ ]:
plotResults(lasso_WMT, X_WMT_test_scaled, y_WMT_test, figsize=(15, 7), asset_name='WMT')


#### 5.3 توقعات CAT


In [ ]:
X_CAT_train = prepareData(adj_close.CAT_Adj_close)
X_CAT_train_scaled, X_CAT_test_scaled, y_CAT_train, y_CAT_test = feature_scaling(adj_close.CAT_Adj_close)

In [ ]:
[(el[0].shape, el[1].shape) for el in time_split.split(X_CAT_train_scaled)]


#### 5.3.1 كات ريدج


In [ ]:
ridge_CAT = grid_search(estimator = Ridge(),
                      X=X_CAT_train_scaled, 
                      y=y_CAT_train,
                      grid_param=param_ridge, 
                      cv=time_split)

In [ ]:
plotResults(ridge_CAT, X_CAT_test_scaled, y_CAT_test, figsize=(15, 7), asset_name='CAT')


#### 5.3.2 كات لاسو


In [ ]:
lasso_CAT = grid_search(estimator = Lasso(),
                      X=X_CAT_train_scaled, 
                      y=y_CAT_train,
                      grid_param=param_lasso,
                      cv=time_split)

In [ ]:
plotResults(lasso_CAT, X_CAT_test_scaled, y_CAT_test, figsize=(15, 7), asset_name='CAT')


### 5.4. توقعات بي بي


In [ ]:
X_BP_train = prepareData(adj_close.BP_Adj_close)
X_BP_train_scaled, X_BP_test_scaled, y_BP_train, y_BP_test = feature_scaling(adj_close.BP_Adj_close)

In [ ]:
[(el[0].shape, el[1].shape) for el in time_split.split(X_CAT_train_scaled)]


#### 5.4.1 بي بي ريدج


In [ ]:
ridge_BP = grid_search(estimator = Ridge(),
                      X=X_BP_train_scaled, 
                      y=y_BP_train,
                      grid_param=param_ridge)

In [ ]:
plotResults(ridge_BP, X_BP_test_scaled, y_BP_test, figsize=(15, 7), asset_name='BP')


#### 5.4.2 BP لاسو


In [ ]:
lasso_BP = grid_search(estimator = Lasso(),
                      X=X_BP_train_scaled, 
                      y=y_BP_train,
                      grid_param=param_lasso,
                      cv=time_split)

In [ ]:
plotResults(lasso_BP, X_BP_test_scaled, y_BP_test, figsize=(15, 7), asset_name='BP')


### توقعات 5.5 فهرنهايت


In [ ]:
X_F_train = prepareData(adj_close.F_Adj_close)
X_F_train_scaled, X_F_test_scaled, y_F_train, y_F_test = feature_scaling(adj_close.F_Adj_close)

In [ ]:
[(el[0].shape, el[1].shape) for el in time_split.split(X_F_train_scaled)]


#### 5.5.1 إف ريدج


In [ ]:
ridge_F = grid_search(estimator = Ridge(),
                      X=X_F_train_scaled, 
                      y=y_F_train,
                      grid_param=param_ridge)

In [ ]:
plotResults(ridge_F, X_F_test_scaled, y_F_test, figsize=(15, 7), asset_name='F')


#### 5.5.2 واو لاسو


In [ ]:
lasso_F = grid_search(estimator = Lasso(),
                      X=X_F_train_scaled, 
                      y=y_F_train,
                      grid_param=param_lasso,
                      cv=time_split)

In [ ]:
plotResults(lasso_F, X_F_test_scaled, y_F_test, figsize=(15, 7), asset_name='F')


###5.6. توقعات جي.اي


In [ ]:
X_GE_train = prepareData(adj_close.GE_Adj_close)
X_GE_train_scaled, X_GE_test_scaled, y_GE_train, y_GE_test = feature_scaling(adj_close.GE_Adj_close)


#### 5.6.1 جي إي ريدج


In [ ]:
ridge_GE = grid_search(estimator = Ridge(),
                      X=X_GE_train_scaled, 
                      y=y_GE_train,
                      grid_param=param_ridge)

In [ ]:
plotResults(ridge_GE, X_GE_test_scaled, y_GE_test, figsize=(15, 7), asset_name='GE')


#### 5.6.2 جي إي لاسو


In [ ]:
lasso_GE = grid_search(estimator = Lasso(),
                      X=X_GE_train_scaled, 
                      y=y_GE_train,
                      grid_param=param_lasso,
                      cv=time_split)

In [ ]:
plotResults(lasso_GE, X_GE_test_scaled, y_GE_test, figsize=(15, 7), asset_name='GE')


## 6. الاستنتاج



في هذا المشروع، قدمنا تحليلًا رسوميًا لشركة United Continental Holdings Inc. (UAL)، وBP p.l.c. (BP)، وشركة American Water Works Company Inc. (AWK)، وشركة Ford Motor Company (F)، وشركة General Electric (GE)، وشركة Walmart Inc. (WMT) بالإضافة إلى أسعار الإغلاق المعدلة. تم أيضًا ملاءمة نموذج ARMA مع البيانات بشكل فردي لنموذج العوائد بالإضافة إلى تطبيق خوارزميات Ridge وLasso ML للتنبؤ بالأسعار. ومع ذلك، هناك طرق عديدة لتحسين وتعزيز التوقعات.1. مزج خوارزميات ML للتنبؤ بما إذا كان السعر سيرتفع أم ينخفض. 
2. يجب أن تؤدي النمذجة المتزامنة لـ ARMA-GARCH إلى تحسين النتيجة بشكل كبير نظرًا لأنه، كما ناقشنا أعلاه، هناك تقلبات متجمعة لا يمكن أن تشملها ARMA وحدها. ومع ذلك، إذا قمنا ببساطة بتطبيق GARCH على بقايا ARMA، فلن يكون النهج متسقًا ذاتيًا.
3. يلزم ضبط الميزات بشكل قوي قبل تطبيق خوارزميات ML. يمكن إضافة السعر المفتوح كميزة إضافية للتنبؤ بأسعار الإغلاق المعدلة.
4. إضافة ميزات إضافية. إحدى الطرق الممكنة لإنشاء ميزة إضافية هي التنبؤ بسعر الإغلاق المعدل لمؤشر NASDAQ أو مؤشر S&P500، واستخدامه كمعلمة إدخال إضافية أثناء ضبط خوارزمية تعلم الآلة.
5. هنا تم تصميم 6 أصول بشكل فردي بينما يمكن تصميمها معًا من أجل تحسين جودة التنبؤات وتقليل إجمالي المخاطر من خلال بناء محفظة. يعد هذا المشروع هو الخطوة الأولى نحو بناء واختبار أداء المحفظة بناءً على هذه الأصول.
6. عندما يتم تحليل الأحداث المتطرفة ويتم تقييم المحفظة، يمكن للمرء أن يقترح استراتيجية التداول. في الوقت الحالي، على الرغم من أنه وفقًا لمقاييس النسبة المئوية المطلقة، قد تبدو تنبؤاتنا جيدة، إذا استخدمناها كتنبؤات إشارة، فإن النتائج ليست واعدة.